In [ ]:
# ============================================================
# 소비패턴 데이터 탐정 - Pandas 분석 (Q1, Q2)
# Q1. 어떤 카테고리에 지출이 가장 많이 몰리는가? (금액, 건수, 비중)
# Q2. 큰 금액 거래는 어떤 특징이 있는가? (기준: 150,000원 초과)
# ============================================================
import pandas as pd                 # 표(DataFrame)를 다루는 라이브러리
import matplotlib.pyplot as plt     # 차트를 그리는 라이브러리

In [ ]:
# ------------------------------------------------------------
# 0. 데이터 불러오기 + 기본 점검
# ------------------------------------------------------------
categories = pd.read_csv("data/categories.csv")   # 카테고리 코드표
customers = pd.read_csv("data/customers.csv")     # 고객 정보
merchants = pd.read_csv("data/merchants.csv")     # 가맹점 정보
df = pd.read_csv("data/transactions.csv")         # 거래내역 (이번 분석의 주인공)
 
print(df.shape)                    # (행 수, 열 수) -> (1200, 9)면 정상
df.info()                          # 열별 자료형, 비어있지 않은 값 개수(결측 확인)
print(df["amount"].describe())     # 금액의 평균, 중앙값, 최대 등 요약

In [ ]:
# ------------------------------------------------------------
# Q1. 카테고리별 지출 (금액, 건수, 비중)
# ------------------------------------------------------------
# 같은 카테고리끼리 묶어서, 금액 합계(total)와 거래 건수(count)를 구함
summary = df.groupby("category")["amount"].agg(total="sum", count="count")
 
# 비중(%) = 카테고리 합계 / 전체 합계 * 100, 소수 첫째 자리까지 반올림
summary["ratio"] = (summary["total"] / summary["total"].sum() * 100).round(1)
 
# 합계가 큰 카테고리가 위로 오도록 정렬
summary = summary.sort_values("total", ascending=False)
print(summary)

In [ ]:
# ----- Q1 차트: 카테고리별 지출 비중 막대 그래프 -----
plt.rcParams["font.family"] = "Malgun Gothic"   # 한글 깨짐 방지 (Windows 기본 폰트)
plt.rcParams["axes.unicode_minus"] = False      # 마이너스 기호 깨짐 방지
 
plt.figure(figsize=(10, 6))                     # 그림 크기 (가로 10, 세로 6)
plt.bar(summary.index, summary["ratio"])        # x: 카테고리 이름, y: 비중(%)
plt.title("카테고리별 지출 비중(%)")
plt.xlabel("카테고리")
plt.ylabel("비중(%)")
plt.xticks(rotation=45)                         # 카테고리 이름이 안 겹치게 기울임
plt.tight_layout()                              # 글자가 잘리지 않게 여백 자동 조정
plt.savefig("category_ratio.png")               # 이미지로 저장 (show보다 먼저!)
plt.show()                                      # 화면에 표시

In [ ]:
# ------------------------------------------------------------
# Q2. 큰 금액 거래 (팀 기준: 150,000원 초과)
# ------------------------------------------------------------
THRESHOLD = 150000    # 기준 금액. 바뀌면 이 한 줄만 고치면 됨
 
# 금액이 기준보다 큰 행만 골라서 복사본으로 저장 (.copy()는 독립된 표로 만들기)
big = df[df["amount"] > THRESHOLD].copy()
print(len(big))              # 고액 거래 건수 -> 55
print(big["amount"].sum())   # 고액 거래 금액 합계 -> 18028000
 
# 금액이 큰 순서로 정렬해서 상위 10건 확인
top10 = big.sort_values("amount", ascending=False).head(10)
print(top10[["date", "category", "amount", "channel"]])   # 필요한 열만 골라서 출력

In [ ]:
# ----- (a) 고액 거래의 카테고리별 특징 -----
big_cat = big.groupby("category")["amount"].agg(total="sum", count="count")
big_cat["ratio"] = (big_cat["total"] / big_cat["total"].sum() * 100).round(1)
big_cat = big_cat.sort_values("total", ascending=False)
print(big_cat)

In [ ]:
# ----- (b) 고액 거래의 월별 특징 -----
# date는 "2026-04-22" 같은 문자열이라, 앞 7글자("2026-04")만 잘라 월 열을 만듦
big["month"] = big["date"].str[:7]
big_month = big.groupby("month")["amount"].agg(total="sum", count="count")
print(big_month)

In [ ]:
# ----- (c) 고액 거래의 채널별 특징 -----
print(big["channel"].value_counts())   # 채널별로 몇 건인지 셈

In [ ]:
# ------------------------------------------------------------
# 인사이트에 쓴 숫자 검증 (문장의 숫자가 코드로 확인되는지 점검)
# ------------------------------------------------------------
print(summary["total"].sum())                                                   # 전체 합계 -> 54656600
print((summary["total"] / summary["count"]).round(0))                           # 카테고리별 건당 평균
print(round(summary["total"].head(3).sum() / summary["total"].sum() * 100, 1))  # 상위 3개 비중 -> 58.7
print(round(summary.loc["FOOD", "count"] / len(df) * 100, 1))                   # 식비 건수 비중 -> 27.0
print(round(len(big) / len(df) * 100, 1))                                       # 고액 건수 비중 -> 4.6
print(round(big["amount"].sum() / df["amount"].sum() * 100, 1))                 # 고액 금액 비중 -> 33.0
 
# 카테고리별 고액 비율 = 그 카테고리 고액 건수 / 그 카테고리 전체 건수
# (두 표가 카테고리 이름으로 자동으로 맞춰져서 나눠짐)
big_rate = (big_cat["count"] / summary["count"] * 100).round(1)
print(big_rate.sort_values(ascending=False))                                    # 구독이 맨 위 -> 11.6

In [ ]:
# ------------------------------------------------------------
# SQL 결과와 비교할 수치표 저장
# ------------------------------------------------------------
# utf-8-sig: 엑셀로 열어도 한글이 깨지지 않게 하는 인코딩
summary.to_csv("pandas_q1_category.csv", encoding="utf-8-sig")
big_cat.to_csv("pandas_q2_big_category.csv", encoding="utf-8-sig")
big_month.to_csv("pandas_q2_big_month.csv", encoding="utf-8-sig")

In [ ]:
# ------------------------------------------------------------
# [핵심 인사이트]
# ------------------------------------------------------------
# 지출은 상위 3개 카테고리에 집중된다. 장보기(21.3%), 쇼핑(19.3%), 식비(18.0%)가 전체 지출 약 5,466만 원의 58.7%를 차지한다. 식비는 건수가 324건(27.0%)으로 가장 많지만 건당 평균이 약 30,403원이라 금액 순위는 3위다.
# 고액 거래(15만 원 초과)는 건수는 적지만 금액 비중이 크다. 55건으로 전체의 4.6%지만 금액으로는 18,028,000원, 전체의 33.0%를 차지한다. 고액 거래는 식비 13건(23.2%)이 가장 많고, 구독 10건(16.3%)이 뒤를 잇는다.
# 시기와 카테고리별 고액 비율에 차이가 있다. 고액 거래는 3월에 2건뿐이고 나머지 달은 7~13건이다. 카테고리 안에서 고액 거래가 차지하는 비율은 구독이 11.6%(86건 중 10건)로 가장 높다.